In [1]:
import shutil
import os
import sys
from lerobot.common.datasets.lerobot_dataset import LEROBOT_HOME
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
# import tensorflow_datasets as tfds
import h5py
import torch
import tqdm
import numpy as np
import time
import glob
from math import sqrt
from matplotlib import pyplot as plt

In [2]:
def cal_cmd_vel(cmd_eef_pos, delta = 3):
    diff = cmd_eef_pos[delta:] - cmd_eef_pos[:-delta]
    return np.sqrt(np.sum(diff*diff, axis=1)).mean()

def cal_cmd_state_diff(cmd_eef_pos, state_eef_pos):
    diff = cmd_eef_pos - state_eef_pos
    return np.sqrt(np.sum(diff*diff, axis=1)).mean()

In [8]:
person = 'LJ'
datapaths = [
                # '/mnt/pfs-chihiro/20250306',
                # '/mnt/pfs-chihiro/20250307',
                # '/mnt/pfs-chihiro/20250308',
                # '/mnt/pfs-chihiro/20250310',
                '/mnt/pfs-chihiro/20250311',
                '/mnt/pfs-chihiro/20250312',
                # '/mnt/pfs-chihiro/20250313',
                # '/mnt/pfs-chihiro/20250314',
                # '/mnt/pfs-chihiro/20250315',
                # '/mnt/pfs-chihiro/20250317',
                # '/mnt/pfs-chihiro/20250318',
]
# mission = '01FULL_HF'
mission = 'QUICK'


In [17]:
# dataset_files = []
# for dataset_path in datapaths:
#     dataset_files += [os.path.join(dataset_path, p) for p in os.listdir(dataset_path) if (person in p and mission in p)]

dataset_files = ['/mnt/pfs-chihiro/20250506/20250506_Y_M105_DYFV2_STEP0to1_HF_GY']
dataset_files = ['/mnt/pfs-chihiro/20250509/20250509_Y_M104_MULTI_PICKPLACE_PutPlateOnRack_LYB']
stats = {'len':[], 'cmd_vel':[], 'cmd_state_diff':[]}
for h5_path in dataset_files:
    hdf5_file_names = glob.glob(os.path.join(h5_path,'*.hdf5'))
    for hdf5_file_name in tqdm.tqdm(hdf5_file_names):
        h5_file = h5py.File(hdf5_file_name,'r')
        ep = {k:np.array(v) for k,v in h5_file.items()}
        h5_file.close()
        
        stats['len'].append(ep['robot0_eef_pos'].shape[0])
        cmd_vel = (cal_cmd_vel(ep['robot0_cmd_eef_pos']) + cal_cmd_vel(ep['robot1_cmd_eef_pos']))/2
        stats['cmd_vel'].append(cmd_vel)
        cmd_state_diff = (cal_cmd_state_diff(ep['robot0_cmd_eef_pos'], ep['robot0_eef_pos']) + 
                          cal_cmd_state_diff(ep['robot1_cmd_eef_pos'], ep['robot1_eef_pos']))/2
        stats['cmd_state_diff'].append(cmd_state_diff)

100%|██████████| 111/111 [01:21<00:00,  1.37it/s]


In [11]:
LTJ_stats = {k:np.array(v) for k,v in stats.items()}
LTJ_stats['cmd_vel'] = np.nan_to_num(LTJ_stats['cmd_vel'], nan=0)
print(LTJ_stats['len'].mean(), LTJ_stats['cmd_vel'].mean()*100, LTJ_stats['cmd_state_diff'].mean()*100)

732.047619047619 1.8471541821819464 4.096450225417531


In [12]:
WW_stats = {k:np.array(v) for k,v in stats.items()}
WW_stats['cmd_vel'] = np.nan_to_num(WW_stats['cmd_vel'], nan=0)
print(WW_stats['len'].mean(), WW_stats['cmd_vel'].mean()*100, WW_stats['cmd_state_diff'].mean()*100)

732.047619047619 1.8471541821819464 4.096450225417531


In [13]:
WHJ_stats = {k:np.array(v) for k,v in stats.items()}
WHJ_stats['cmd_vel'] = np.nan_to_num(WHJ_stats['cmd_vel'], nan=0)
print(WHJ_stats['len'].mean(), WHJ_stats['cmd_vel'].mean()*100, WHJ_stats['cmd_state_diff'].mean()*100)

732.047619047619 1.8471541821819464 4.096450225417531


In [14]:
# 01FULL:
540.184375 1.5836817739552236 1.2352673762587456 LTJ
547.596 2.026993309712263 1.3055635018067329 WW
658.5813953488372 2.1277687988182152 1.034181737937435 WHJ
542.9243697478992 2.4095494553841497 1.6220656764659982 GY
423.9474835886214 2.32668148990611 1.8547732562473669 LJ
576.2321755027423 1.8712744577388145 1.2054597370946096 GZQ

# 15QUICK:
1681.741935483871 1.1485287221484533 0.8118516881422249 WHJ 
1740.2863849765258 1.1173638447983816 1.100564652226307 LJ


SyntaxError: invalid syntax (1119072037.py, line 2)

In [1]:
import os
import cv2
import numpy as np
from lerobot.common.datasets.lerobot_dataset import LEROBOT_HOME
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
import ffmpeg
import ipdb
import torch
import tqdm
import matplotlib.pyplot as plt
from random import sample 
def plot_robot_xyplane_stats(action_points, state_points):
    points_np = action_points.numpy()[...,:2]
    points_np2 = state_points.numpy()[...,:2]
    # Plot
    plt.figure(figsize=(8, 6))
    plt.scatter(points_np[:, 0], points_np[:, 1],
                s=2,               # dot size
                c='red',          # dot color
                alpha=.1)        # transparency to emphasize density
    plt.scatter(points_np[:, 0], points_np[:, 1],
                s=2,               # dot size
                c='blue',          # dot color
                alpha=.1)        # transparency to emphasize density
    plt.title('Robot eef on XY plane')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.axis('equal')
    plt.grid(True)
    plt.show()
    # plt.savefig(f'/root/PI_Official/data/visualization/{save_name}.png', dpi=300, bbox_inches='tight')

def cal_cmd_vel(cmd_eef_pos, delta = 3):
    diff = cmd_eef_pos[delta:] - cmd_eef_pos[:-delta]
    return np.sqrt(np.sum(diff*diff, axis=1)).mean()

def cal_cmd_state_diff(action_points, state_points):
    action_points = action_points.numpy()
    state_points = state_points.numpy()
    diff = action_points - state_points
    return np.sqrt(np.sum(diff*diff, axis=1))

In [2]:
repo_id='YC_MeetingRoom_PutPenInBox_0515'
sample_num = 20_000

dataset = LeRobotDataset(repo_id, local_files_only=True)
action_points = []
state_points = []
for i in tqdm.tqdm(sample(range(len(dataset)),sample_num)):

    item = dataset[i]
    action_points.append(item['actions'][:3])
    action_points.append(item['actions'][7:10])
    state_points.append(item['observation.state'][:3])
    state_points.append(item['observation.state'][7:10])

action_points = torch.stack(action_points)
state_points = torch.stack(state_points)

Returning existing local_dir `/pfstem/likaiyu/resources/lerobot/YC_MeetingRoom_PutPenInBox_0515` as remote repo cannot be accessed in `snapshot_download` (None).
Returning existing local_dir `/pfstem/likaiyu/resources/lerobot/YC_MeetingRoom_PutPenInBox_0515` as remote repo cannot be accessed in `snapshot_download` (None).


Resolving data files:   0%|          | 0/376 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/169 [00:00<?, ?it/s]

100%|██████████| 20000/20000 [03:06<00:00, 107.05it/s]


In [2]:
plot_robot_xyplane_stats(action_points, state_points)

NameError: name 'plot_robot_xyplane_stats' is not defined

In [43]:
sum(action_points == torch.tensor([0,0]))


tensor([6, 6])

In [1]:
cmd_state_diff = cal_cmd_state_diff(action_points, state_points)
sum(cmd_state_diff > 0.1)

NameError: name 'cal_cmd_state_diff' is not defined